<a href="https://colab.research.google.com/github/HK25abm/Transformer-Phishing-Detection/blob/main/Phase2_Adversarial_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [6]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("DEVICE:", DEVICE)

DEVICE: cuda


In [7]:
PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

TRAIN_PATH = os.path.join(
    PROJECT_DIR,
    "train.csv"
)

VALIDATION_PATH = os.path.join(
    PROJECT_DIR,
    "validation.csv"
)

TEST_PATH = os.path.join(
    PROJECT_DIR,
    "test.csv"
)

BERT_PATH = os.path.join(
    PROJECT_DIR,
    "saved_models",
    "bert"
)

ROBERTA_PATH = os.path.join(
    PROJECT_DIR,
    "saved_models",
    "roberta"
)

DEBERTA_PATH = os.path.join(
    PROJECT_DIR,
    "saved_models",
    "deberta_fixed"
)

PHASE2_DIR = os.path.join(
    PROJECT_DIR,
    "phase2_adversarial_training"
)

os.makedirs(
    PHASE2_DIR,
    exist_ok=True
)

print("Phase 2 folder:")
print(PHASE2_DIR)

Phase 2 folder:
/content/drive/MyDrive/phishing_project/phase2_adversarial_training


In [8]:
train_df = pd.read_csv(
    TRAIN_PATH
)

validation_df = pd.read_csv(
    VALIDATION_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)

for df in [
    train_df,
    validation_df,
    test_df
]:

    df["text"] = (
        df["text"]
        .fillna("")
        .astype(str)
    )

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce"
    )

    df.dropna(
        subset=["label"],
        inplace=True
    )

    df["label"] = (
        df["label"]
        .astype(int)
    )

In [9]:
print("TRAIN")
print(train_df["label"].value_counts())

print("\nVALIDATION")
print(validation_df["label"].value_counts())

print("\nTEST")
print(test_df["label"].value_counts())

TRAIN
label
0    7000
1    7000
Name: count, dtype: int64

VALIDATION
label
1    1500
0    1500
Name: count, dtype: int64

TEST
label
0    1500
1    1500
Name: count, dtype: int64


In [10]:
print("TRAIN")
print(train_df["label"].value_counts())

print("\nVALIDATION")
print(validation_df["label"].value_counts())

print("\nTEST")
print(test_df["label"].value_counts())

TRAIN
label
0    7000
1    7000
Name: count, dtype: int64

VALIDATION
label
1    1500
0    1500
Name: count, dtype: int64

TEST
label
0    1500
1    1500
Name: count, dtype: int64


In [11]:
print(
    train_df.columns.tolist()
)

['text', 'label', 'source', 'email_type', 'urls', 'sample_id']


In [12]:
print(
    "Unique IDs:",
    train_df["sample_id"].nunique()
)

print(
    "Rows:",
    len(train_df)
)

Unique IDs: 14000
Rows: 14000


In [13]:
if "sample_id" not in train_df.columns:

    train_df = train_df.reset_index(
        drop=True
    )

    train_df["sample_id"] = [
        f"train_{i}"
        for i in range(
            len(train_df)
        )
    ]

print(
    train_df[
        [
            "sample_id",
            "label"
        ]
    ].head()
)

      sample_id  label
0  EMAIL_016400      0
1  EMAIL_004650      0
2  EMAIL_013860      1
3  EMAIL_015476      0
4  EMAIL_017077      1


In [14]:
train_phishing_df = train_df[
    train_df["label"] == 1
].copy()

print(
    "Training phishing samples:",
    len(train_phishing_df)
)

Training phishing samples: 7000


In [15]:
ADVERSARIAL_TRAIN_RATIO = 0.20

In [16]:
n_adversarial_train = int(
    len(train_phishing_df)
    * ADVERSARIAL_TRAIN_RATIO
)

adv_source_df = (
    train_phishing_df
    .sample(
        n=n_adversarial_train,
        random_state=SEED
    )
    .copy()
)

print(
    "Clean phishing training samples:",
    len(train_phishing_df)
)

print(
    "Selected for adversarial generation:",
    len(adv_source_df)
)

Clean phishing training samples: 7000
Selected for adversarial generation: 1400


In [17]:
adv_source_df[
    [
        "sample_id",
        "label"
    ]
].to_csv(
    os.path.join(
        PHASE2_DIR,
        "phase2_adversarial_training_source_ids.csv"
    ),
    index=False
)

print(
    "Phase 2 source IDs saved."
)

Phase 2 source IDs saved.


In [18]:
bert_tokenizer = AutoTokenizer.from_pretrained(
    BERT_PATH
)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_PATH,
    torch_dtype=torch.float32
)

bert_model.to(DEVICE)
bert_model.eval()

print("Frozen BERT baseline loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Frozen BERT baseline loaded.


In [19]:
FINAL_ATTACK_CONFIG = {
    "max_changes": 4,
    "top_important_words": 15,
    "mlm_top_k": 30,
    "similarity_threshold": 0.92
}

In [20]:
adv_train_pilot_source = (
    adv_source_df
    .sample(
        n=50,
        random_state=SEED
    )
    .copy()
)

print(
    "Pilot size:",
    len(adv_train_pilot_source)
)

Pilot size: 50


In [21]:
adv_train_pilot_results = []

for counter, (_, row) in enumerate(
    adv_train_pilot_source.iterrows(),
    start=1
):

    print(
        f"Training attack "
        f"{counter}/"
        f"{len(adv_train_pilot_source)}"
    )

    result = contextual_adversarial_attack_v2(
        text=row["text"],
        model=bert_model,
        tokenizer=bert_tokenizer,
        max_changes=FINAL_ATTACK_CONFIG["max_changes"],
        top_important_words=FINAL_ATTACK_CONFIG["top_important_words"],
        mlm_top_k=FINAL_ATTACK_CONFIG["mlm_top_k"],
        similarity_threshold=FINAL_ATTACK_CONFIG["similarity_threshold"]
    )

    result["sample_id"] = row["sample_id"]
    result["label"] = 1

    adv_train_pilot_results.append(
        result
    )

adv_train_pilot_df = pd.DataFrame(
    adv_train_pilot_results
)

Training attack 1/50


NameError: name 'contextual_adversarial_attack_v2' is not defined

In [22]:
functions = [
    "contextual_adversarial_attack_v2",
    "get_probabilities",
    "get_probabilities_batch",
    "get_important_words",
    "get_mlm_replacements",
    "semantic_similarity",
    "replace_word_once",
    "valid_source_word",
    "valid_replacement"
]

for f in functions:
    print(
        f,
        "FOUND" if f in globals() else "MISSING"
    )

contextual_adversarial_attack_v2 MISSING
get_probabilities MISSING
get_probabilities_batch MISSING
get_important_words MISSING
get_mlm_replacements MISSING
semantic_similarity MISSING
replace_word_once MISSING
valid_source_word MISSING
valid_replacement MISSING


In [23]:
!pip install -q sentence-transformers

In [24]:
import re
import os
import random
import numpy as np
import pandas as pd
import torch

from transformers import pipeline
from sentence_transformers import SentenceTransformer

In [25]:
mlm_name = "google-bert/bert-base-uncased"

fill_mask = pipeline(
    "fill-mask",
    model=mlm_name,
    tokenizer=mlm_name,
    device=0 if torch.cuda.is_available() else -1
)

MLM_MASK = fill_mask.tokenizer.mask_token

similarity_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=str(DEVICE)
)

print("MLM and similarity models loaded.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MLM and similarity models loaded.


In [26]:
FINAL_ATTACK_CONFIG = {
    "max_changes": 4,
    "top_important_words": 15,
    "mlm_top_k": 30,
    "similarity_threshold": 0.92
}

print(FINAL_ATTACK_CONFIG)

{'max_changes': 4, 'top_important_words': 15, 'mlm_top_k': 30, 'similarity_threshold': 0.92}


In [27]:
STOP_WORDS = {
    "the", "a", "an", "and", "or", "but",
    "if", "then", "to", "of", "in", "on",
    "at", "for", "with", "this", "that",
    "these", "those", "is", "are", "was",
    "were", "be", "been", "being", "you",
    "your", "we", "our", "i", "he", "she",
    "they", "it", "as", "by", "from"
}

PROTECTED_WORDS = {
    "http", "https", "www",
    "com", "net", "org",
    "html", "php", "aspx"
}

In [28]:
def split_words(text):

    return re.findall(
        r"\b[A-Za-z][A-Za-z'-]*\b",
        str(text)
    )


def remove_word_once(text, word):

    pattern = re.compile(
        r"\b" + re.escape(word) + r"\b",
        flags=re.IGNORECASE
    )

    return pattern.sub(
        "",
        str(text),
        count=1
    )


def replace_word_once(
    text,
    old_word,
    new_word
):

    pattern = re.compile(
        r"\b" + re.escape(old_word) + r"\b",
        flags=re.IGNORECASE
    )

    return pattern.sub(
        new_word,
        str(text),
        count=1
    )


def mask_word_once(text, word):

    pattern = re.compile(
        r"\b" + re.escape(word) + r"\b",
        flags=re.IGNORECASE
    )

    return pattern.sub(
        MLM_MASK,
        str(text),
        count=1
    )

In [29]:
def is_protected_word(word):

    word_lower = word.lower()

    if word_lower in PROTECTED_WORDS:
        return True

    if len(word_lower) < 4:
        return True

    if any(
        char.isdigit()
        for char in word
    ):
        return True

    if not word.isalpha():
        return True

    return False


def valid_source_word(word):

    if not word.isalpha():
        return False

    if any(
        char.isupper()
        for char in word[1:]
    ):
        return False

    if len(word) < 4:
        return False

    if len(word) > 20:
        return False

    return True

In [30]:
def valid_replacement(
    original_word,
    replacement
):

    original = original_word.lower()
    candidate = replacement.lower()

    if not candidate.isalpha():
        return False

    if len(candidate) < 4:
        return False

    if candidate == original:
        return False

    if candidate.startswith("##"):
        return False

    if candidate in PROTECTED_WORDS:
        return False

    ratio = (
        len(candidate)
        / max(len(original), 1)
    )

    if (
        ratio < 0.75
        or ratio > 1.35
    ):
        return False

    GENERIC_BAD = {
        "milk",
        "called",
        "ultra",
        "some",
        "from",
        "during",
        "ones",
        "assistant"
    }

    if candidate in GENERIC_BAD:
        return False

    return True

In [31]:
def get_probabilities_batch(
    model,
    tokenizer,
    texts,
    batch_size=8,
    max_length=256
):

    model.eval()

    all_probs = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        encoding = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoding = {
            k: v.to(DEVICE)
            for k, v in encoding.items()
        }

        with torch.no_grad():

            logits = model(
                **encoding
            ).logits

            probs = torch.softmax(
                logits.float(),
                dim=1
            )

        all_probs.append(
            probs.cpu().numpy()
        )

    return np.concatenate(
        all_probs,
        axis=0
    )

In [32]:
def semantic_similarity(
    text1,
    text2
):

    embeddings = similarity_model.encode(
        [text1, text2],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    return float(
        np.dot(
            embeddings[0],
            embeddings[1]
        )
    )

In [33]:
def get_mlm_replacements(
    text,
    word,
    top_k=30
):

    masked_text = mask_word_once(
        text,
        word
    )

    if MLM_MASK not in masked_text:
        return []

    try:

        outputs = fill_mask(
            masked_text,
            top_k=top_k,
            tokenizer_kwargs={
                "truncation": True,
                "max_length": 256
            }
        )

    except Exception:
        return []

    replacements = []

    for result in outputs:

        candidate = (
            result["token_str"]
            .strip()
        )

        candidate = re.sub(
            r"[^A-Za-z]",
            "",
            candidate
        )

        if not valid_replacement(
            word,
            candidate
        ):
            continue

        replacements.append(
            candidate
        )

    return list(
        dict.fromkeys(
            replacements
        )
    )

In [34]:
def get_important_words(
    text,
    model,
    tokenizer,
    top_k=15
):

    original_prob = get_probabilities_batch(
        model,
        tokenizer,
        [text],
        batch_size=1
    )[0, 1]

    words = split_words(
        text
    )

    candidate_words = []
    seen = set()

    for word in words:

        lower = word.lower()

        if (
            len(lower) < 4
            or lower in STOP_WORDS
            or lower in seen
            or is_protected_word(word)
            or not valid_source_word(word)
        ):
            continue

        seen.add(lower)
        candidate_words.append(word)

    candidate_words = candidate_words[:50]

    if len(candidate_words) == 0:
        return []

    modified_texts = [
        remove_word_once(
            text,
            word
        )
        for word in candidate_words
    ]

    probs = get_probabilities_batch(
        model,
        tokenizer,
        modified_texts,
        batch_size=8
    )[:, 1]

    importance = []

    for word, probability in zip(
        candidate_words,
        probs
    ):

        importance.append({
            "word": word,
            "importance": float(
                original_prob
                - probability
            )
        })

    importance = sorted(
        importance,
        key=lambda x: x["importance"],
        reverse=True
    )

    return importance[:top_k]

In [35]:
def contextual_adversarial_attack_v2(
    text,
    model,
    tokenizer,
    max_changes=4,
    top_important_words=15,
    mlm_top_k=30,
    similarity_threshold=0.92
):

    original_text = str(text)

    original_prob = float(
        get_probabilities_batch(
            model,
            tokenizer,
            [original_text],
            batch_size=1
        )[0, 1]
    )

    current_text = original_text
    current_prob = original_prob

    changes = []

    important_words = get_important_words(
        current_text,
        model,
        tokenizer,
        top_k=top_important_words
    )

    for item in important_words:

        if len(changes) >= max_changes:
            break

        word = item["word"]

        if is_protected_word(word):
            continue

        replacements = get_mlm_replacements(
            current_text,
            word,
            top_k=mlm_top_k
        )

        if not replacements:
            continue

        candidate_texts = []
        candidate_info = []

        for replacement in replacements:

            candidate_text = replace_word_once(
                current_text,
                word,
                replacement
            )

            similarity = semantic_similarity(
                original_text,
                candidate_text
            )

            if similarity < similarity_threshold:
                continue

            candidate_texts.append(
                candidate_text
            )

            candidate_info.append({
                "replacement":
                    replacement,
                "similarity":
                    similarity
            })

        if not candidate_texts:
            continue

        probabilities = get_probabilities_batch(
            model,
            tokenizer,
            candidate_texts,
            batch_size=8
        )[:, 1]

        best_index = int(
            np.argmin(
                probabilities
            )
        )

        best_prob = float(
            probabilities[
                best_index
            ]
        )

        if best_prob < current_prob:

            best_candidate = candidate_texts[
                best_index
            ]

            best_info = candidate_info[
                best_index
            ]

            changes.append({
                "old":
                    word,
                "new":
                    best_info[
                        "replacement"
                    ],
                "before_prob":
                    current_prob,
                "after_prob":
                    best_prob,
                "similarity":
                    best_info[
                        "similarity"
                    ]
            })

            current_text = best_candidate
            current_prob = best_prob

        if current_prob < 0.5:
            break

    final_similarity = semantic_similarity(
        original_text,
        current_text
    )

    return {
        "original_text":
            original_text,
        "adversarial_text":
            current_text,
        "original_probability":
            original_prob,
        "adversarial_probability":
            current_prob,
        "confidence_drop":
            original_prob - current_prob,
        "semantic_similarity":
            final_similarity,
        "num_changes":
            len(changes),
        "changes":
            changes,
        "successful_evasion":
            (
                original_prob >= 0.5
                and
                current_prob < 0.5
                and
                final_similarity
                >= similarity_threshold
            )
    }

In [36]:
required_functions = [
    "contextual_adversarial_attack_v2",
    "get_probabilities_batch",
    "get_important_words",
    "get_mlm_replacements",
    "semantic_similarity",
    "replace_word_once",
    "valid_source_word",
    "valid_replacement"
]

for fn in required_functions:

    print(
        fn,
        "OK"
        if fn in globals()
        else "MISSING"
    )

contextual_adversarial_attack_v2 OK
get_probabilities_batch OK
get_important_words OK
get_mlm_replacements OK
semantic_similarity OK
replace_word_once OK
valid_source_word OK
valid_replacement OK


In [37]:
adv_train_pilot_results = []

for counter, (_, row) in enumerate(
    adv_train_pilot_source.iterrows(),
    start=1
):

    print(
        f"Training attack "
        f"{counter}/"
        f"{len(adv_train_pilot_source)}"
    )

    result = contextual_adversarial_attack_v2(
        text=row["text"],
        model=bert_model,
        tokenizer=bert_tokenizer,
        max_changes=
            FINAL_ATTACK_CONFIG[
                "max_changes"
            ],
        top_important_words=
            FINAL_ATTACK_CONFIG[
                "top_important_words"
            ],
        mlm_top_k=
            FINAL_ATTACK_CONFIG[
                "mlm_top_k"
            ],
        similarity_threshold=
            FINAL_ATTACK_CONFIG[
                "similarity_threshold"
            ]
    )

    result["sample_id"] = (
        row["sample_id"]
    )

    result["label"] = 1

    adv_train_pilot_results.append(
        result
    )

adv_train_pilot_df = pd.DataFrame(
    adv_train_pilot_results
)

Training attack 1/50
Training attack 2/50


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Training attack 3/50
Training attack 4/50
Training attack 5/50
Training attack 6/50
Training attack 7/50
Training attack 8/50
Training attack 9/50
Training attack 10/50
Training attack 11/50
Training attack 12/50
Training attack 13/50
Training attack 14/50
Training attack 15/50
Training attack 16/50
Training attack 17/50
Training attack 18/50
Training attack 19/50
Training attack 20/50
Training attack 21/50
Training attack 22/50
Training attack 23/50
Training attack 24/50
Training attack 25/50
Training attack 26/50
Training attack 27/50
Training attack 28/50
Training attack 29/50
Training attack 30/50
Training attack 31/50
Training attack 32/50
Training attack 33/50
Training attack 34/50
Training attack 35/50
Training attack 36/50
Training attack 37/50
Training attack 38/50
Training attack 39/50
Training attack 40/50
Training attack 41/50
Training attack 42/50
Training attack 43/50
Training attack 44/50
Training attack 45/50
Training attack 46/50
Training attack 47/50
Training attack 4

In [38]:
print(
    "Mean confidence drop:",
    adv_train_pilot_df[
        "confidence_drop"
    ].mean()
)

print(
    "Mean semantic similarity:",
    adv_train_pilot_df[
        "semantic_similarity"
    ].mean()
)

print(
    "Mean substitutions:",
    adv_train_pilot_df[
        "num_changes"
    ].mean()
)

print(
    "Successful evasions:",
    adv_train_pilot_df[
        "successful_evasion"
    ].sum()
)

print(
    "Modified samples:",
    (
        adv_train_pilot_df[
            "num_changes"
        ] > 0
    ).sum(),
    "/",
    len(
        adv_train_pilot_df
    )
)

Mean confidence drop: 0.03269585051224567
Mean semantic similarity: 0.9525705230236053
Mean substitutions: 3.46
Successful evasions: 2
Modified samples: 49 / 50


In [39]:
adv_train_results = []

TOTAL = len(adv_source_df)

for counter, (_, row) in enumerate(
    adv_source_df.iterrows(),
    start=1
):

    if counter % 25 == 0:
        print(f"{counter}/{TOTAL}")

    result = contextual_adversarial_attack_v2(
        text=row["text"],
        model=bert_model,
        tokenizer=bert_tokenizer,
        max_changes=FINAL_ATTACK_CONFIG["max_changes"],
        top_important_words=FINAL_ATTACK_CONFIG["top_important_words"],
        mlm_top_k=FINAL_ATTACK_CONFIG["mlm_top_k"],
        similarity_threshold=FINAL_ATTACK_CONFIG["similarity_threshold"]
    )

    result["sample_id"] = row["sample_id"]
    result["label"] = 1
    result["source"] = row["source"]
    result["email_type"] = row["email_type"]

    adv_train_results.append(result)

adv_train_df = pd.DataFrame(adv_train_results)

25/1400
50/1400
75/1400
100/1400
125/1400
150/1400
175/1400
200/1400
225/1400
250/1400
275/1400
300/1400
325/1400
350/1400
375/1400
400/1400
425/1400
450/1400
475/1400
500/1400
525/1400
550/1400
575/1400
600/1400
625/1400
650/1400
675/1400
700/1400
725/1400
750/1400
775/1400
800/1400
825/1400
850/1400
875/1400
900/1400
925/1400
950/1400
975/1400
1000/1400
1025/1400
1050/1400
1075/1400
1100/1400
1125/1400
1150/1400
1175/1400
1200/1400
1225/1400
1250/1400
1275/1400
1300/1400
1325/1400
1350/1400
1375/1400
1400/1400


In [40]:
SAVE_PATH = os.path.join(
    PHASE2_DIR,
    "phase2_adversarial_training_examples.csv"
)

adv_train_df.to_csv(
    SAVE_PATH,
    index=False
)

print("Saved:", SAVE_PATH)

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/phase2_adversarial_training_examples.csv


In [41]:
print("Generated:", len(adv_train_df))

print("Modified:",
      (adv_train_df["num_changes"] > 0).sum())

print("Successful evasions:",
      adv_train_df["successful_evasion"].sum())

print("Mean confidence drop:",
      adv_train_df["confidence_drop"].mean())

print("Mean semantic similarity:",
      adv_train_df["semantic_similarity"].mean())

print("Mean substitutions:",
      adv_train_df["num_changes"].mean())

Generated: 1400
Modified: 1354
Successful evasions: 35
Mean confidence drop: 0.023151976694716723
Mean semantic similarity: 0.9542468702367374
Mean substitutions: 3.4778571428571428


In [42]:
sample = adv_train_df.sample(
    20,
    random_state=42
)

for _, row in sample.iterrows():

    print("=" * 80)

    print("ORIGINAL:\n")
    print(row["original_text"])

    print("\nADVERSARIAL:\n")
    print(row["adversarial_text"])

    print("\nSimilarity:",
          round(row["semantic_similarity"],4))

    print("Changes:",
          row["changes"])

ORIGINAL:

is ROLEX under 199 $ good for you? Replica Rolex Watches ! SAVE Big ! http://nievetubaelay.com/

ADVERSARIAL:

is ROLEX under 199 $ good for you? please Rolex replica ! SAVE Big ! http://nievetubaelay.com/

Similarity: 0.9301
Changes: [{'old': 'Watches', 'new': 'replica', 'before_prob': 0.9999380111694336, 'after_prob': 0.9999299049377441, 'similarity': 0.9319082498550415}, {'old': 'Replica', 'new': 'please', 'before_prob': 0.9999299049377441, 'after_prob': 0.9999252557754517, 'similarity': 0.9300543665885925}]
ORIGINAL:

Dear Friend:Find solutions to all your daily problems and life's challenges at the click of a mouse button?We have the answers you're looking for on The Word Bible CD-ROM it is one of the most powerful, life-changing tools available today and it's easy to use.On one CD, (Windows or Macintosh versions) you have a complete library of Bibles, well known reference books and study tools. You can view several Bible versions simultaneously, make personal notes, pr

In [43]:
adv_phishing = adv_train_df.copy()

adv_phishing = adv_phishing.rename(
    columns={
        "adversarial_text":"text"
    }
)

adv_phishing = adv_phishing[
    [
        "sample_id",
        "text",
        "label",
        "source",
        "email_type"
    ]
]

In [44]:
augmented_train_df = pd.concat(
    [
        train_df,
        adv_phishing
    ],
    ignore_index=True
)

In [45]:
augmented_train_df = augmented_train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [46]:
print(augmented_train_df.label.value_counts())

print()

print("Rows:",len(augmented_train_df))

label
1    8400
0    7000
Name: count, dtype: int64

Rows: 15400


In [47]:
SAVE = os.path.join(
    PHASE2_DIR,
    "augmented_training_dataset.csv"
)

augmented_train_df.to_csv(
    SAVE,
    index=False
)

print("Saved:",SAVE)

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/augmented_training_dataset.csv


In [48]:
valid_adv_train_df = adv_train_df[
    (adv_train_df["num_changes"] > 0)
    &
    (adv_train_df["semantic_similarity"] >= 0.92)
].copy()

print("Generated:", len(adv_train_df))
print("Valid modified:", len(valid_adv_train_df))

print(
    "Mean similarity:",
    valid_adv_train_df[
        "semantic_similarity"
    ].mean()
)

print(
    "Mean substitutions:",
    valid_adv_train_df[
        "num_changes"
    ].mean()
)

Generated: 1400
Valid modified: 1354
Mean similarity: 0.9526924791152932
Mean substitutions: 3.5960118168389954


In [49]:
print(
    valid_adv_train_df[
        "label"
    ].value_counts()
)

label
1    1354
Name: count, dtype: int64


In [50]:
valid_adv_train_df[
    "parent_sample_id"
] = valid_adv_train_df[
    "sample_id"
]

valid_adv_train_df[
    "sample_id"
] = (
    "ADV_"
    +
    valid_adv_train_df[
        "parent_sample_id"
    ].astype(str)
)

In [51]:
display(
    valid_adv_train_df[
        [
            "sample_id",
            "parent_sample_id",
            "num_changes",
            "semantic_similarity"
        ]
    ].head()
)

,sample_id,parent_sample_id,num_changes,semantic_similarity
0,ADV_EMAIL_006705,EMAIL_006705,1,0.925523
1,ADV_EMAIL_014097,EMAIL_014097,4,0.969739
4,ADV_EMAIL_000812,EMAIL_000812,4,0.928327
5,ADV_EMAIL_003791,EMAIL_003791,4,0.924150
6,ADV_EMAIL_005852,EMAIL_005852,4,0.925134


In [52]:
adv_phishing_df = pd.DataFrame({

    "text":
        valid_adv_train_df[
            "adversarial_text"
        ],

    "label":
        1,

    "source":
        "adversarial_training",

    "email_type":
        "adversarial_phishing",

    "urls":
        "",

    "sample_id":
        valid_adv_train_df[
            "sample_id"
        ]
})

In [53]:
print(
    adv_phishing_df.shape
)

display(
    adv_phishing_df.head()
)

(1354, 6)


,text,label,source,email_type,urls,sample_id
0,Pe ug nis En ek till tj ment: Does It Work? Or...,1,adversarial_training,adversarial_phishing,,ADV_EMAIL_006705
1,123 You wouldn’t believe it but it became real...,1,adversarial_training,adversarial_phishing,,ADV_EMAIL_014097
4,retail Large-volume dealer of diabetic supplie...,1,adversarial_training,adversarial_phishing,,ADV_EMAIL_000812
5,Many surveys have noted that ladies prefer eac...,1,adversarial_training,adversarial_phishing,,ADV_EMAIL_003791
6,very important ! read carefully # # # # # # # ...,1,adversarial_training,adversarial_phishing,,ADV_EMAIL_005852


In [54]:
augmented_train_df = pd.concat(
    [
        train_df,
        adv_phishing_df
    ],
    ignore_index=True
)

augmented_train_df = (
    augmented_train_df
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(
        drop=True
    )
)

print(
    "Augmented training size:",
    len(augmented_train_df)
)

print(
    "\nClass distribution:"
)

print(
    augmented_train_df[
        "label"
    ].value_counts()
)

Augmented training size: 15354

Class distribution:
label
1    8354
0    7000
Name: count, dtype: int64


In [55]:
train_ids = set(
    augmented_train_df[
        "sample_id"
    ].astype(str)
)

test_ids = set(
    test_df[
        "sample_id"
    ].astype(str)
)

overlap = train_ids.intersection(
    test_ids
)

print(
    "Train/Test ID overlap:",
    len(overlap)
)

Train/Test ID overlap: 0


In [56]:
parent_ids = set(
    valid_adv_train_df[
        "parent_sample_id"
    ].astype(str)
)

test_ids = set(
    test_df[
        "sample_id"
    ].astype(str)
)

print(
    "Adversarial parent/test overlap:",
    len(
        parent_ids.intersection(
            test_ids
        )
    )
)

Adversarial parent/test overlap: 0


In [57]:
parent_ids = set(
    valid_adv_train_df[
        "parent_sample_id"
    ].astype(str)
)

test_ids = set(
    test_df[
        "sample_id"
    ].astype(str)
)

print(
    "Adversarial parent/test overlap:",
    len(
        parent_ids.intersection(
            test_ids
        )
    )
)

Adversarial parent/test overlap: 0


In [58]:
AUGMENTED_TRAIN_PATH = os.path.join(
    PHASE2_DIR,
    "augmented_training_dataset.csv"
)

augmented_train_df.to_csv(
    AUGMENTED_TRAIN_PATH,
    index=False
)

print(
    "Saved:",
    AUGMENTED_TRAIN_PATH
)

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/augmented_training_dataset.csv


In [59]:
VALID_ADV_PATH = os.path.join(
    PHASE2_DIR,
    "validated_adversarial_training_examples.csv"
)

valid_adv_train_df.to_csv(
    VALID_ADV_PATH,
    index=False
)

print(
    "Saved:",
    VALID_ADV_PATH
)

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/validated_adversarial_training_examples.csv


In [60]:
class EmailDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=256
    ):

        self.texts = (
            pd.Series(texts)
            .fillna("")
            .astype(str)
            .tolist()
        )

        self.labels = (
            pd.Series(labels)
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):

        return len(
            self.labels
        )

    def __getitem__(
        self,
        idx
    ):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length
        )

        encoding[
            "labels"
        ] = self.labels[idx]

        return encoding

In [61]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def compute_metrics_adv_training(
    eval_pred
):

    logits, labels = eval_pred

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    logits = (
        logits
        -
        np.max(
            logits,
            axis=1,
            keepdims=True
        )
    )

    exp_logits = np.exp(
        logits
    )

    probs = (
        exp_logits
        /
        np.sum(
            exp_logits,
            axis=1,
            keepdims=True
        )
    )

    predictions = np.argmax(
        probs,
        axis=1
    )

    return {

        "accuracy":
            accuracy_score(
                labels,
                predictions
            ),

        "precision":
            precision_score(
                labels,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                predictions,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                labels,
                probs[:, 1]
            )
    }

In [62]:
if "fill_mask" in globals():
    del fill_mask

if "similarity_model" in globals():
    del similarity_model

if "bert_model" in globals():
    del bert_model

if "bert_tokenizer" in globals():
    del bert_tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU memory cleared.")

GPU memory cleared.


In [63]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

bert_adv_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BERT_PATH
    )
)

bert_adv_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        BERT_PATH,
        torch_dtype=torch.float32
    )
)

bert_adv_model.to(
    DEVICE
)

print(
    "Phase 1 BERT baseline loaded."
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Phase 1 BERT baseline loaded.


In [64]:
bert_adv_train_dataset = EmailDataset(
    augmented_train_df["text"],
    augmented_train_df["label"],
    bert_adv_tokenizer,
    max_length=256
)

bert_adv_validation_dataset = EmailDataset(
    validation_df["text"],
    validation_df["label"],
    bert_adv_tokenizer,
    max_length=256
)

bert_clean_test_dataset = EmailDataset(
    test_df["text"],
    test_df["label"],
    bert_adv_tokenizer,
    max_length=256
)

In [65]:
bert_adv_collator = (
    DataCollatorWithPadding(
        tokenizer=
            bert_adv_tokenizer
    )
)

print(
    "Training:",
    len(bert_adv_train_dataset)
)

print(
    "Validation:",
    len(bert_adv_validation_dataset)
)

print(
    "Test:",
    len(bert_clean_test_dataset)
)

Training: 15354
Validation: 3000
Test: 3000


In [66]:
BERT_DEFENDED_PATH = os.path.join(
    PHASE2_DIR,
    "saved_models",
    "bert_adversarially_trained"
)

BERT_CHECKPOINT_DIR = (
    "/content/"
    "bert_adversarial_training_checkpoints"
)

os.makedirs(
    BERT_DEFENDED_PATH,
    exist_ok=True
)

In [67]:
bert_adv_training_args = TrainingArguments(

    output_dir=
        BERT_CHECKPOINT_DIR,

    learning_rate=
        1e-5,

    per_device_train_batch_size=
        8,

    gradient_accumulation_steps=
        2,

    per_device_eval_batch_size=
        16,

    num_train_epochs=
        2,

    weight_decay=
        0.01,

    warmup_ratio=
        0.10,

    max_grad_norm=
        1.0,

    eval_strategy=
        "epoch",

    save_strategy=
        "epoch",

    logging_strategy=
        "epoch",

    load_best_model_at_end=
        True,

    metric_for_best_model=
        "f1",

    greater_is_better=
        True,

    save_total_limit=
        1,

    report_to=
        "none",

    seed=
        SEED,

    data_seed=
        SEED,

    fp16=
        False,

    bf16=
        False
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [68]:
bert_adv_trainer = Trainer(

    model=
        bert_adv_model,

    args=
        bert_adv_training_args,

    train_dataset=
        bert_adv_train_dataset,

    eval_dataset=
        bert_adv_validation_dataset,

    processing_class=
        bert_adv_tokenizer,

    data_collator=
        bert_adv_collator,

    compute_metrics=
        compute_metrics_adv_training,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [69]:
import time

start_time = time.time()

bert_adv_train_output = (
    bert_adv_trainer.train()
)

bert_adv_training_time = (
    time.time()
    -
    start_time
)

print(
    "Adversarial BERT training time:",
    round(
        bert_adv_training_time,
        2
    ),
    "seconds"
)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.042091,0.092175,0.986000,0.991903,0.980000,0.985915,0.998374
2,0.010943,0.101155,0.985667,0.988598,0.982667,0.985624,0.998398


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Adversarial BERT training time: 1569.45 seconds


In [70]:
bert_def_clean_output = (
    bert_adv_trainer.predict(
        bert_clean_test_dataset
    )
)

bert_def_clean_logits = (
    bert_def_clean_output.predictions
)

bert_def_clean_true = (
    bert_def_clean_output.label_ids
)

bert_def_clean_probs = (
    torch.softmax(
        torch.tensor(
            bert_def_clean_logits
        ).float(),
        dim=1
    )
    .numpy()
)

bert_def_clean_pred = np.argmax(
    bert_def_clean_probs,
    axis=1
)

In [71]:
bert_def_clean_metrics = {

    "model":
        "BERT",

    "condition":
        "Clean_After_Defence",

    "accuracy":
        accuracy_score(
            bert_def_clean_true,
            bert_def_clean_pred
        ),

    "precision":
        precision_score(
            bert_def_clean_true,
            bert_def_clean_pred,
            zero_division=0
        ),

    "recall":
        recall_score(
            bert_def_clean_true,
            bert_def_clean_pred,
            zero_division=0
        ),

    "f1":
        f1_score(
            bert_def_clean_true,
            bert_def_clean_pred,
            zero_division=0
        ),

    "roc_auc":
        roc_auc_score(
            bert_def_clean_true,
            bert_def_clean_probs[:, 1]
        )
}

print(
    bert_def_clean_metrics
)

{'model': 'BERT', 'condition': 'Clean_After_Defence', 'accuracy': 0.9846666666666667, 'precision': 0.9879194630872483, 'recall': 0.9813333333333333, 'f1': 0.9846153846153847, 'roc_auc': np.float64(0.9987542222222223)}


In [72]:
bert_adv_trainer.model.save_pretrained(
    BERT_DEFENDED_PATH
)

bert_adv_tokenizer.save_pretrained(
    BERT_DEFENDED_PATH
)

print(
    "Defended BERT saved:",
    BERT_DEFENDED_PATH
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Defended BERT saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/bert_adversarially_trained


In [73]:
pd.DataFrame(
    [bert_def_clean_metrics]
).to_csv(
    os.path.join(
        PHASE2_DIR,
        "bert_defended_clean_metrics.csv"
    ),
    index=False
)

In [74]:
# 1. Evaluate the in-memory trainer model again
bert_best_eval = bert_adv_trainer.evaluate()

print(bert_best_eval)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.010943,0.092082,2,0.986000,0.991903,0.980000,0.985915,0.998378


{'eval_loss': 0.0920819491147995, 'eval_accuracy': 0.986, 'eval_precision': 0.9919028340080972, 'eval_recall': 0.98, 'eval_f1': 0.9859154929577465, 'eval_roc_auc': 0.9983782222222222}


In [75]:
BERT_DEFENDED_PATH = os.path.join(
    PHASE2_DIR,
    "saved_models",
    "bert_adversarially_trained"
)

os.makedirs(
    BERT_DEFENDED_PATH,
    exist_ok=True
)

bert_adv_trainer.model.save_pretrained(
    BERT_DEFENDED_PATH
)

bert_adv_tokenizer.save_pretrained(
    BERT_DEFENDED_PATH
)

print("Saved:", BERT_DEFENDED_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/bert_adversarially_trained


In [76]:
BERT_DEFENDED_PATH = os.path.join(
    PHASE2_DIR,
    "saved_models",
    "bert_adversarially_trained"
)

os.makedirs(
    BERT_DEFENDED_PATH,
    exist_ok=True
)

bert_adv_trainer.model.save_pretrained(
    BERT_DEFENDED_PATH
)

bert_adv_tokenizer.save_pretrained(
    BERT_DEFENDED_PATH
)

print("Saved:", BERT_DEFENDED_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/bert_adversarially_trained


In [77]:
del bert_adv_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

bert_def_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_DEFENDED_PATH,
    torch_dtype=torch.float32
)

bert_def_tokenizer = AutoTokenizer.from_pretrained(
    BERT_DEFENDED_PATH
)

bert_def_model.to(DEVICE)
bert_def_model.eval()

print("Reloaded defended BERT.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reloaded defended BERT.


In [78]:
samples = [
    "Your PayPal account has been suspended. Verify your password immediately.",
    "The meeting has been moved to Tuesday afternoon.",
    "Cheap cialis xanax online pharmacy. Click here now."
]

for text in samples:

    probs = get_probabilities_batch(
        bert_def_model,
        bert_def_tokenizer,
        [text],
        batch_size=1
    )[0]

    print("\n", text)
    print(
        "Legitimate:",
        round(float(probs[0]), 6),
        "| Phishing:",
        round(float(probs[1]), 6)
    )


 Your PayPal account has been suspended. Verify your password immediately.
Legitimate: 5.9e-05 | Phishing: 0.999941

 The meeting has been moved to Tuesday afternoon.
Legitimate: 0.999975 | Phishing: 2.4e-05

 Cheap cialis xanax online pharmacy. Click here now.
Legitimate: 3.9e-05 | Phishing: 0.999961


In [79]:
BERT_PHASE1_ADV_PATH = os.path.join(
    PROJECT_DIR,
    "phase1_contextual_adversarial",
    "bert_contextual_adversarial_results.csv"
)

bert_phase1_adv_df = pd.read_csv(
    BERT_PHASE1_ADV_PATH
)

print(
    bert_phase1_adv_df.columns.tolist()
)

print(
    "Rows:",
    len(bert_phase1_adv_df)
)


['original_text', 'adversarial_text', 'original_probability', 'adversarial_probability', 'confidence_drop', 'semantic_similarity', 'num_changes', 'changes', 'successful_evasion', 'sample_id']
Rows: 300


In [80]:
def build_model_adversarial_test(
    original_test_df,
    attack_results_df
):
    adv_test = original_test_df.copy()

    attack_map = dict(
        zip(
            attack_results_df["sample_id"],
            attack_results_df["adversarial_text"]
        )
    )

    adv_test["text"] = adv_test.apply(
        lambda row: (
            attack_map.get(
                row["sample_id"],
                row["text"]
            )
            if row["label"] == 1
            else row["text"]
        ),
        axis=1
    )

    return adv_test

In [81]:
bert_frozen_adv_test_df = build_model_adversarial_test(
    test_df,
    bert_phase1_adv_df
)

print(
    bert_frozen_adv_test_df[
        "label"
    ].value_counts()
)

label
0    1500
1    1500
Name: count, dtype: int64


In [82]:
bert_defended_adv_dataset = EmailDataset(
    bert_frozen_adv_test_df["text"],
    bert_frozen_adv_test_df["label"],
    bert_def_tokenizer,
    max_length=256
)

In [83]:
def evaluate_dataset_model(
    model,
    tokenizer,
    dataframe,
    batch_size=16
):

    probs = get_probabilities_batch(
        model,
        tokenizer,
        dataframe["text"]
        .fillna("")
        .astype(str)
        .tolist(),
        batch_size=batch_size
    )

    y_true = (
        dataframe["label"]
        .astype(int)
        .values
    )

    y_pred = np.argmax(
        probs,
        axis=1
    )

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probs[:, 1]
            )
    }

In [84]:
bert_defended_adv_metrics = evaluate_dataset_model(
    bert_def_model,
    bert_def_tokenizer,
    bert_frozen_adv_test_df,
    batch_size=16
)

print(
    bert_defended_adv_metrics
)

{'accuracy': 0.9793333333333333, 'precision': 0.9877883310719131, 'recall': 0.9706666666666667, 'f1': 0.9791526563550773, 'roc_auc': np.float64(0.9982857777777778)}


In [85]:
bert_phase2_comparison = pd.DataFrame(
    [
        {
            "condition": "Baseline_Clean",
            "accuracy": 0.9873,
            "recall": 0.9873,
            "f1": 0.9873,
            "roc_auc": 0.9989
        },
        {
            "condition": "Baseline_Adversarial",
            "accuracy": 0.9823,
            "recall": 0.9773,
            "f1": 0.9822,
            "roc_auc": 0.9985
        },
        {
            "condition": "Defended_Clean",
            "accuracy": 0.9846666667,
            "recall": 0.9813333333,
            "f1": 0.9846153846,
            "roc_auc": 0.9987542222
        },
        {
            "condition": "Defended_Adversarial",
            "accuracy":
                bert_defended_adv_metrics[
                    "accuracy"
                ],
            "recall":
                bert_defended_adv_metrics[
                    "recall"
                ],
            "f1":
                bert_defended_adv_metrics[
                    "f1"
                ],
            "roc_auc":
                bert_defended_adv_metrics[
                    "roc_auc"
                ]
        }
    ]
)

display(
    bert_phase2_comparison.round(4)
)

,condition,accuracy,recall,f1,roc_auc
0,Baseline_Clean,0.9873,0.9873,0.9873,0.9989
1,Baseline_Adversarial,0.9823,0.9773,0.9822,0.9985
2,Defended_Clean,0.9847,0.9813,0.9846,0.9988
3,Defended_Adversarial,0.9793,0.9707,0.9792,0.9983


In [86]:
bert_robustness_gain = (
    bert_defended_adv_metrics["recall"]
    - 0.9773
)

print(
    "BERT robustness gain in adversarial recall:",
    round(
        bert_robustness_gain,
        4
    )
)

BERT robustness gain in adversarial recall: -0.0066


In [87]:
TRIAL1_DIR = os.path.join(
    PHASE2_DIR,
    "trial1_bert"
)

os.makedirs(
    TRIAL1_DIR,
    exist_ok=True
)

bert_phase2_comparison.to_csv(
    os.path.join(
        TRIAL1_DIR,
        "bert_trial1_comparison.csv"
    ),
    index=False
)

print("Trial 1 saved.")

Trial 1 saved.


In [88]:
hard_adv_df = (
    valid_adv_train_df
    .sort_values(
        "confidence_drop",
        ascending=False
    )
    .head(500)
    .copy()
)

print(
    "Hard adversarial examples:",
    len(hard_adv_df)
)

print(
    "Successful evasions included:",
    hard_adv_df[
        "successful_evasion"
    ].sum()
)

print(
    "Mean confidence drop:",
    hard_adv_df[
        "confidence_drop"
    ].mean()
)

print(
    "Mean similarity:",
    hard_adv_df[
        "semantic_similarity"
    ].mean()
)

Hard adversarial examples: 500
Successful evasions included: 35
Mean confidence drop: 0.06477486304854392
Mean similarity: 0.9529428066015243


In [89]:
hard_adv_phishing_df = pd.DataFrame({

    "text":
        hard_adv_df[
            "adversarial_text"
        ],

    "label": 1,

    "source":
        "hard_adversarial_training",

    "email_type":
        "adversarial_phishing",

    "urls": "",

    "sample_id":
        (
            "HARD_ADV_"
            +
            hard_adv_df[
                "sample_id"
            ].astype(str)
        )
})

In [90]:
augmented_train_v2 = pd.concat(
    [
        train_df,
        hard_adv_phishing_df
    ],
    ignore_index=True
)

augmented_train_v2 = (
    augmented_train_v2
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(drop=True)
)

print(
    augmented_train_v2[
        "label"
    ].value_counts()
)

print(
    "Total:",
    len(augmented_train_v2)
)

label
1    7500
0    7000
Name: count, dtype: int64
Total: 14500


In [91]:
del bert_def_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

bert_v2_tokenizer = AutoTokenizer.from_pretrained(
    BERT_PATH
)

bert_v2_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_PATH,
    torch_dtype=torch.float32
)

bert_v2_model.to(DEVICE)

print(
    "Original Phase 1 BERT reloaded."
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Original Phase 1 BERT reloaded.


In [92]:
bert_v2_train_dataset = EmailDataset(
    augmented_train_v2["text"],
    augmented_train_v2["label"],
    bert_v2_tokenizer,
    max_length=256
)

bert_v2_validation_dataset = EmailDataset(
    validation_df["text"],
    validation_df["label"],
    bert_v2_tokenizer,
    max_length=256
)

bert_v2_collator = DataCollatorWithPadding(
    tokenizer=bert_v2_tokenizer
)

In [93]:
BERT_V2_CHECKPOINT_DIR = (
    "/content/bert_adv_trial2_checkpoints"
)

bert_v2_args = TrainingArguments(

    output_dir=
        BERT_V2_CHECKPOINT_DIR,

    learning_rate=
        5e-6,

    per_device_train_batch_size=
        8,

    gradient_accumulation_steps=
        2,

    per_device_eval_batch_size=
        16,

    num_train_epochs=
        1,

    weight_decay=
        0.01,

    warmup_ratio=
        0.05,

    max_grad_norm=
        1.0,

    eval_strategy=
        "epoch",

    save_strategy=
        "epoch",

    logging_strategy=
        "epoch",

    load_best_model_at_end=
        True,

    metric_for_best_model=
        "f1",

    greater_is_better=
        True,

    save_total_limit=
        1,

    report_to=
        "none",

    seed=
        42,

    data_seed=
        42,

    fp16=
        False,

    bf16=
        False
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [94]:
bert_v2_trainer = Trainer(

    model=
        bert_v2_model,

    args=
        bert_v2_args,

    train_dataset=
        bert_v2_train_dataset,

    eval_dataset=
        bert_v2_validation_dataset,

    processing_class=
        bert_v2_tokenizer,

    data_collator=
        bert_v2_collator,

    compute_metrics=
        compute_metrics_adv_training
)

In [95]:
bert_v2_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.027005,0.098106,0.985333,0.986631,0.984000,0.985314,0.998455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=907, training_loss=0.027004795011375284, metrics={'train_runtime': 756.633, 'train_samples_per_second': 19.164, 'train_steps_per_second': 1.199, 'total_flos': 1892148765735600.0, 'train_loss': 0.027004795011375284, 'epoch': 1.0})

In [96]:
BERT_TRIAL2_PATH = os.path.join(
    PHASE2_DIR,
    "saved_models",
    "bert_trial2"
)

os.makedirs(BERT_TRIAL2_PATH, exist_ok=True)

bert_v2_trainer.model.save_pretrained(BERT_TRIAL2_PATH)
bert_v2_tokenizer.save_pretrained(BERT_TRIAL2_PATH)

print("Saved:", BERT_TRIAL2_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/bert_trial2


In [97]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

trial2_tokenizer = AutoTokenizer.from_pretrained(
    BERT_TRIAL2_PATH
)

trial2_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_TRIAL2_PATH
)

trial2_model.to(DEVICE)
trial2_model.eval()

print("Trial 2 model loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trial 2 model loaded.


In [98]:
clean_metrics = evaluate_dataset_model(
    trial2_model,
    trial2_tokenizer,
    test_df,
    batch_size=16
)

print(clean_metrics)

{'accuracy': 0.987, 'precision': 0.9847378898473789, 'recall': 0.9893333333333333, 'f1': 0.9870302627203192, 'roc_auc': np.float64(0.9989657777777778)}


In [99]:
adv_metrics = evaluate_dataset_model(
    trial2_model,
    trial2_tokenizer,
    bert_frozen_adv_test_df,
    batch_size=16
)

print(adv_metrics)

{'accuracy': 0.9833333333333333, 'precision': 0.9846256684491979, 'recall': 0.982, 'f1': 0.9833110814419226, 'roc_auc': np.float64(0.9986435555555556)}


In [100]:
trial2_results = pd.DataFrame([

{
"Condition":"Baseline Clean",
"Accuracy":0.9873,
"Recall":0.9873,
"F1":0.9873,
"ROC_AUC":0.9989
},

{
"Condition":"Baseline Adversarial",
"Accuracy":0.9823,
"Recall":0.9773,
"F1":0.9822,
"ROC_AUC":0.9985
},

{
"Condition":"Trial2 Clean",
"Accuracy":clean_metrics["accuracy"],
"Recall":clean_metrics["recall"],
"F1":clean_metrics["f1"],
"ROC_AUC":clean_metrics["roc_auc"]
},

{
"Condition":"Trial2 Adversarial",
"Accuracy":adv_metrics["accuracy"],
"Recall":adv_metrics["recall"],
"F1":adv_metrics["f1"],
"ROC_AUC":adv_metrics["roc_auc"]
}

])

display(trial2_results.round(4))

,Condition,Accuracy,Recall,F1,ROC_AUC
0,Baseline Clean,0.9873,0.9873,0.9873,0.9989
1,Baseline Adversarial,0.9823,0.9773,0.9822,0.9985
2,Trial2 Clean,0.9870,0.9893,0.9870,0.9990
3,Trial2 Adversarial,0.9833,0.9820,0.9833,0.9986


In [101]:
baseline_adv_recall = 0.9773

trial2_adv_recall = adv_metrics["recall"]

gain = trial2_adv_recall - baseline_adv_recall

print("Robustness Gain:", round(gain,4))

Robustness Gain: 0.0047


In [102]:
SAVE_DIR = os.path.join(
    PHASE2_DIR,
    "trial2_results"
)

os.makedirs(SAVE_DIR, exist_ok=True)

trial2_results.to_csv(
    os.path.join(
        SAVE_DIR,
        "bert_trial2_results.csv"
    ),
    index=False
)

pd.DataFrame([clean_metrics]).to_csv(
    os.path.join(
        SAVE_DIR,
        "bert_trial2_clean.csv"
    ),
    index=False
)

pd.DataFrame([adv_metrics]).to_csv(
    os.path.join(
        SAVE_DIR,
        "bert_trial2_adversarial.csv"
    ),
    index=False
)

print("Trial 2 saved.")

Trial 2 saved.


In [103]:
import os

FINAL_BERT_DIR = os.path.join(
    PROJECT_DIR,
    "final_phase2_results",
    "bert_final_defended"
)

os.makedirs(FINAL_BERT_DIR, exist_ok=True)

print(FINAL_BERT_DIR)

/content/drive/MyDrive/phishing_project/final_phase2_results/bert_final_defended


In [104]:
trial2_model.save_pretrained(FINAL_BERT_DIR)
trial2_tokenizer.save_pretrained(FINAL_BERT_DIR)

print("Model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


In [105]:
import pandas as pd

clean_df = pd.DataFrame([clean_metrics])

clean_df.to_csv(
    os.path.join(
        FINAL_BERT_DIR,
        "clean_metrics.csv"
    ),
    index=False
)


In [106]:
adv_df = pd.DataFrame([adv_metrics])

adv_df.to_csv(
    os.path.join(
        FINAL_BERT_DIR,
        "adversarial_metrics.csv"
    ),
    index=False
)

In [107]:
trial2_results.to_csv(
    os.path.join(
        FINAL_BERT_DIR,
        "robustness_comparison.csv"
    ),
    index=False
)

In [108]:
training_time = pd.DataFrame({

    "model":["BERT"],

    "training_type":[
        "Adversarial Training"
    ],

    "epochs":[1],

    "learning_rate":[5e-6],

    "training_time_seconds":[
        bert_adv_training_time
    ]
})

training_time.to_csv(
    os.path.join(
        FINAL_BERT_DIR,
        "training_time.csv"
    ),
    index=False
)

In [109]:
clean_probs = get_probabilities_batch(
    trial2_model,
    trial2_tokenizer,
    test_df["text"].tolist(),
    batch_size=16
)

clean_pred = clean_probs.argmax(axis=1)

clean_predictions = test_df.copy()

clean_predictions["prediction"] = clean_pred

clean_predictions["prob_legitimate"] = clean_probs[:,0]

clean_predictions["prob_phishing"] = clean_probs[:,1]

clean_predictions.to_csv(
    os.path.join(
        FINAL_BERT_DIR,
        "clean_predictions.csv"
    ),
    index=False
)

In [110]:
adv_probs = get_probabilities_batch(
    trial2_model,
    trial2_tokenizer,
    bert_frozen_adv_test_df["text"].tolist(),
    batch_size=16
)

adv_pred = adv_probs.argmax(axis=1)

adv_predictions = bert_frozen_adv_test_df.copy()

adv_predictions["prediction"] = adv_pred

adv_predictions["prob_legitimate"] = adv_probs[:,0]

adv_predictions["prob_phishing"] = adv_probs[:,1]

adv_predictions.to_csv(
    os.path.join(
        FINAL_BERT_DIR,
        "adversarial_predictions.csv"
    ),
    index=False
)

In [111]:
from sklearn.metrics import confusion_matrix
import numpy as np

clean_cm = confusion_matrix(
    test_df["label"],
    clean_pred
)

adv_cm = confusion_matrix(
    bert_frozen_adv_test_df["label"],
    adv_pred
)

np.savetxt(
    os.path.join(
        FINAL_BERT_DIR,
        "clean_confusion_matrix.csv"
    ),
    clean_cm,
    fmt="%d",
    delimiter=","
)

np.savetxt(
    os.path.join(
        FINAL_BERT_DIR,
        "adversarial_confusion_matrix.csv"
    ),
    adv_cm,
    fmt="%d",
    delimiter=","
)

In [112]:
from sklearn.metrics import roc_curve

fpr_clean, tpr_clean, _ = roc_curve(
    test_df["label"],
    clean_probs[:,1]
)

pd.DataFrame({

    "FPR":fpr_clean,

    "TPR":tpr_clean

}).to_csv(

    os.path.join(
        FINAL_BERT_DIR,
        "clean_roc.csv"
    ),
    index=False
)

fpr_adv, tpr_adv, _ = roc_curve(

    bert_frozen_adv_test_df["label"],

    adv_probs[:,1]
)

pd.DataFrame({

    "FPR":fpr_adv,

    "TPR":tpr_adv

}).to_csv(

    os.path.join(
        FINAL_BERT_DIR,
        "adversarial_roc.csv"
    ),
    index=False
)

In [113]:
summary = pd.DataFrame({

"Model":["BERT"],

"Baseline_Recall":[0.9773],

"Defended_Recall":[0.9820],

"Robustness_Gain":[0.0047],

"Attack":"Contextual MLM",

"Defense":"Adversarial Training"

})

summary.to_csv(

    os.path.join(
        FINAL_BERT_DIR,
        "experiment_summary.csv"
    ),

    index=False
)

In [114]:
for name in [
    "trial2_model",
    "trial2_tokenizer",
    "bert_v2_model",
    "bert_v2_tokenizer",
    "bert_v2_trainer"
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU cleared.")

GPU cleared.


In [115]:
ROBERTA_PATH = os.path.join(
    PROJECT_DIR,
    "saved_models",
    "roberta"
)

roberta_v2_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_PATH
)

roberta_v2_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_PATH,
    torch_dtype=torch.float32
)

roberta_v2_model.to(DEVICE)

print("Original Phase 1 RoBERTa loaded.")

Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Original Phase 1 RoBERTa loaded.


In [116]:
roberta_v2_train_dataset = EmailDataset(
    augmented_train_v2["text"],
    augmented_train_v2["label"],
    roberta_v2_tokenizer,
    max_length=256
)

roberta_v2_validation_dataset = EmailDataset(
    validation_df["text"],
    validation_df["label"],
    roberta_v2_tokenizer,
    max_length=256
)

roberta_clean_test_dataset = EmailDataset(
    test_df["text"],
    test_df["label"],
    roberta_v2_tokenizer,
    max_length=256
)

roberta_v2_collator = DataCollatorWithPadding(
    tokenizer=roberta_v2_tokenizer
)

print("Train:", len(roberta_v2_train_dataset))
print("Validation:", len(roberta_v2_validation_dataset))
print("Test:", len(roberta_clean_test_dataset))

Train: 14500
Validation: 3000
Test: 3000


In [117]:
ROBERTA_CHECKPOINT_DIR = (
    "/content/roberta_adv_trial2_checkpoints"
)

ROBERTA_DEFENDED_PATH = os.path.join(
    PHASE2_DIR,
    "saved_models",
    "roberta_final_defended"
)

os.makedirs(
    ROBERTA_DEFENDED_PATH,
    exist_ok=True
)

In [118]:
roberta_v2_args = TrainingArguments(

    output_dir=
        ROBERTA_CHECKPOINT_DIR,

    learning_rate=
        5e-6,

    per_device_train_batch_size=
        8,

    gradient_accumulation_steps=
        2,

    per_device_eval_batch_size=
        16,

    num_train_epochs=
        1,

    weight_decay=
        0.01,

    warmup_ratio=
        0.05,

    max_grad_norm=
        1.0,

    eval_strategy=
        "epoch",

    save_strategy=
        "epoch",

    logging_strategy=
        "epoch",

    load_best_model_at_end=
        True,

    metric_for_best_model=
        "f1",

    greater_is_better=
        True,

    save_total_limit=
        1,

    report_to=
        "none",

    seed=
        42,

    data_seed=
        42,

    fp16=
        False,

    bf16=
        False
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [119]:
roberta_v2_trainer = Trainer(

    model=
        roberta_v2_model,

    args=
        roberta_v2_args,

    train_dataset=
        roberta_v2_train_dataset,

    eval_dataset=
        roberta_v2_validation_dataset,

    processing_class=
        roberta_v2_tokenizer,

    data_collator=
        roberta_v2_collator,

    compute_metrics=
        compute_metrics_adv_training
)

In [120]:
import time

start_time = time.time()

roberta_v2_trainer.train()

roberta_adv_training_time = (
    time.time()
    -
    start_time
)

print(
    "RoBERTa adversarial training time:",
    round(
        roberta_adv_training_time,
        2
    ),
    "seconds"
)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.026713,0.076630,0.988667,0.993935,0.983333,0.988606,0.999225


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa adversarial training time: 766.14 seconds


In [121]:
roberta_v2_trainer.model.save_pretrained(
    ROBERTA_DEFENDED_PATH
)

roberta_v2_tokenizer.save_pretrained(
    ROBERTA_DEFENDED_PATH
)

print(
    "Saved:",
    ROBERTA_DEFENDED_PATH
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/roberta_final_defended


In [122]:
roberta_def_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        ROBERTA_DEFENDED_PATH,
        torch_dtype=torch.float32
    )
)

roberta_def_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        ROBERTA_DEFENDED_PATH
    )
)

roberta_def_model.to(DEVICE)
roberta_def_model.eval()

print("Reloaded defended RoBERTa.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reloaded defended RoBERTa.


In [123]:
roberta_clean_metrics = evaluate_dataset_model(
    roberta_def_model,
    roberta_def_tokenizer,
    test_df,
    batch_size=16
)

print(
    "Defended RoBERTa clean:"
)

print(
    roberta_clean_metrics
)

Defended RoBERTa clean:
{'accuracy': 0.9906666666666667, 'precision': 0.9946236559139785, 'recall': 0.9866666666666667, 'f1': 0.9906291834002677, 'roc_auc': np.float64(0.9992764444444444)}


In [124]:
ROBERTA_PHASE1_ADV_PATH = os.path.join(
    PROJECT_DIR,
    "phase1_contextual_adversarial",
    "roberta_contextual_adversarial_results.csv"
)

roberta_phase1_adv_df = pd.read_csv(
    ROBERTA_PHASE1_ADV_PATH
)

In [125]:
roberta_frozen_adv_test_df = build_model_adversarial_test(
    test_df,
    roberta_phase1_adv_df
)

print(
    roberta_frozen_adv_test_df[
        "label"
    ].value_counts()
)

label
0    1500
1    1500
Name: count, dtype: int64


In [126]:
roberta_adv_metrics = evaluate_dataset_model(
    roberta_def_model,
    roberta_def_tokenizer,
    roberta_frozen_adv_test_df,
    batch_size=16
)

print(
    "Defended RoBERTa adversarial:"
)

print(
    roberta_adv_metrics
)

Defended RoBERTa adversarial:
{'accuracy': 0.9886666666666667, 'precision': 0.9946018893387314, 'recall': 0.9826666666666667, 'f1': 0.98859825620389, 'roc_auc': np.float64(0.9991697777777778)}


In [127]:
roberta_phase2_comparison = pd.DataFrame([
    {
        "Condition": "Baseline Clean",
        "Accuracy": 0.9890,
        "Recall": 0.9847,
        "F1": 0.9890,
        "ROC_AUC": 0.9992
    },
    {
        "Condition": "Baseline Adversarial",
        "Accuracy": 0.9853,
        "Recall": 0.9773,
        "F1": 0.9852,
        "ROC_AUC": 0.9991
    },
    {
        "Condition": "Defended Clean",
        "Accuracy": roberta_clean_metrics["accuracy"],
        "Recall": roberta_clean_metrics["recall"],
        "F1": roberta_clean_metrics["f1"],
        "ROC_AUC": roberta_clean_metrics["roc_auc"]
    },
    {
        "Condition": "Defended Adversarial",
        "Accuracy": roberta_adv_metrics["accuracy"],
        "Recall": roberta_adv_metrics["recall"],
        "F1": roberta_adv_metrics["f1"],
        "ROC_AUC": roberta_adv_metrics["roc_auc"]
    }
])

display(
    roberta_phase2_comparison.round(4)
)

,Condition,Accuracy,Recall,F1,ROC_AUC
0,Baseline Clean,0.9890,0.9847,0.9890,0.9992
1,Baseline Adversarial,0.9853,0.9773,0.9852,0.9991
2,Defended Clean,0.9907,0.9867,0.9906,0.9993
3,Defended Adversarial,0.9887,0.9827,0.9886,0.9992


In [128]:
roberta_robustness_gain = (
    roberta_adv_metrics["recall"]
    - 0.9773
)

print(
    "RoBERTa robustness gain:",
    round(
        roberta_robustness_gain,
        4
    )
)

RoBERTa robustness gain: 0.0054


In [129]:
FINAL_ROBERTA_DIR = os.path.join(
    PROJECT_DIR,
    "final_phase2_results",
    "roberta_final_defended"
)

os.makedirs(
    FINAL_ROBERTA_DIR,
    exist_ok=True
)

In [130]:
roberta_def_model.save_pretrained(
    FINAL_ROBERTA_DIR
)

roberta_def_tokenizer.save_pretrained(
    FINAL_ROBERTA_DIR
)

print(
    "Saved:",
    FINAL_ROBERTA_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/phishing_project/final_phase2_results/roberta_final_defended


In [131]:
pd.DataFrame(
    [roberta_clean_metrics]
).to_csv(
    os.path.join(
        FINAL_ROBERTA_DIR,
        "clean_metrics.csv"
    ),
    index=False
)

pd.DataFrame(
    [roberta_adv_metrics]
).to_csv(
    os.path.join(
        FINAL_ROBERTA_DIR,
        "adversarial_metrics.csv"
    ),
    index=False
)

In [132]:
roberta_phase2_comparison.to_csv(
    os.path.join(
        FINAL_ROBERTA_DIR,
        "robustness_comparison.csv"
    ),
    index=False
)

In [133]:
roberta_summary = pd.DataFrame(
    [{
        "model": "RoBERTa",
        "baseline_adv_recall": 0.9773,
        "defended_adv_recall":
            roberta_adv_metrics["recall"],
        "robustness_gain":
            roberta_adv_metrics["recall"]
            - 0.9773,
        "baseline_clean_accuracy": 0.9890,
        "defended_clean_accuracy":
            roberta_clean_metrics["accuracy"],
        "baseline_adv_accuracy": 0.9853,
        "defended_adv_accuracy":
            roberta_adv_metrics["accuracy"]
    }]
)

roberta_summary.to_csv(
    os.path.join(
        FINAL_ROBERTA_DIR,
        "experiment_summary.csv"
    ),
    index=False
)

display(
    roberta_summary.round(4)
)

,model,baseline_adv_recall,defended_adv_recall,robustness_gain,baseline_clean_accuracy,defended_clean_accuracy,baseline_adv_accuracy,defended_adv_accuracy
0,RoBERTa,0.9773,0.9827,0.0054,0.989,0.9907,0.9853,0.9887


In [134]:
print(
    os.listdir(
        FINAL_ROBERTA_DIR
    )
)

['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'clean_metrics.csv', 'adversarial_metrics.csv', 'robustness_comparison.csv', 'experiment_summary.csv']


In [ ]:
for name in [
    "roberta_v2_model",
    "roberta_v2_tokenizer",
    "roberta_v2_trainer",
    "roberta_def_model",
    "roberta_def_tokenizer"
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU cleared.")